In [ ]:
import os
import glob
import json
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# ======================================
# CONFIG
# ======================================

DATA_DIR = "../knowledge_base/disease_profiles"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

INDEX_FILE = "faiss_index.bin"
CHUNKS_FILE = "chunks.pkl"

# ======================================
# LOAD JSON -> CHUNKS
# ======================================

all_chunks = []

for file_path in glob.glob(os.path.join(DATA_DIR, "*.json")):

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    disease_id = data.get("disease_id", "UNKNOWN")
    disease_name = data.get("name", disease_id)

    if "overview" in data:
        all_chunks.append({
            "disease": disease_id,
            "type": "overview",
            "text": f"{disease_name}. {data['overview']}"
        })

    if "symptoms" in data:
        all_chunks.append({
            "disease": disease_id,
            "type": "symptoms",
            "text": f"{disease_name}. Symptoms: {', '.join(data['symptoms'])}"
        })

    if "risk_factors" in data:
        all_chunks.append({
            "disease": disease_id,
            "type": "risk_factors",
            "text": f"{disease_name}. Risk factors: {', '.join(data['risk_factors'])}"
        })

    if "treatment" in data:
        all_chunks.append({
            "disease": disease_id,
            "type": "treatment",
            "text": f"{disease_name}. Treatment: {', '.join(data['treatment'])}"
        })

    if "diagnosis" in data:
        all_chunks.append({
            "disease": disease_id,
            "type": "diagnosis",
            "text": f"{disease_name}. Diagnosis: {', '.join(data['diagnosis'])}"
        })

print(f"Total chunks: {len(all_chunks)}")

# ======================================
# EMBEDDING
# ======================================

model = SentenceTransformer(EMBEDDING_MODEL)

texts = [chunk["text"] for chunk in all_chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

print("Embedding shape:", embeddings.shape)

# ======================================
# BUILD FAISS
# ======================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors in FAISS:", index.ntotal)

# ======================================
# SAVE
# ======================================

faiss.write_index(index, INDEX_FILE)

with open(CHUNKS_FILE, "wb") as f:
    pickle.dump(all_chunks, f)

print("FAISS index saved.")

# ======================================
# TEST SEARCH
# ======================================

query = "dark skin lesion with irregular border and color variation"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype(np.float32)

faiss.normalize_L2(query_embedding)

distances, indices = index.search(
    query_embedding,
    k=5
)

print("\n")
print("=" * 80)
print("QUERY:", query)
print("=" * 80)

for rank, idx in enumerate(indices[0], start=1):

    chunk = all_chunks[idx]

    print(f"\nTOP {rank}")
    print("Disease :", chunk["disease"])
    print("Type    :", chunk["type"])
    print("Distance:", round(float(distances[0][rank - 1]), 4))
    print("Text    :", chunk["text"])

Total chunks: 20


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]

Embedding shape: (20, 384)
Vectors in FAISS: 20
FAISS index saved.


QUERY: dark skin lesion with irregular border and color variation

TOP 1
Disease : UNKNOWN
Type    : overview
Distance: 1.0983
Text    : UNKNOWN. {'definition': 'A benign fibrous skin nodule commonly occurring on the extremities.', 'summary': 'Dermatofibroma is a harmless skin lesion often developing after minor trauma or insect bites.'}

TOP 2
Disease : UNKNOWN
Type    : diagnosis
Distance: 1.1228
Text    : UNKNOWN. Diagnosis: clinical_examination, dermoscopy_features, histopathology

TOP 3
Disease : UNKNOWN
Type    : diagnosis
Distance: 1.1228
Text    : UNKNOWN. Diagnosis: clinical_examination, dermoscopy_features, histopathology

TOP 4
Disease : UNKNOWN
Type    : diagnosis
Distance: 1.1228
Text    : UNKNOWN. Diagnosis: clinical_examination, dermoscopy_features, histopathology

TOP 5
Disease : UNKNOWN
Type    : diagnosis
Distance: 1.1228
Text    : UNKNOWN. Diagnosis: clinical_examination, dermoscopy_features, histop

In [7]:
import os
import glob
import json
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# =====================================================
# CONFIG
# =====================================================

DATA_DIR = "../knowledge_base/disease_profiles"

INDEX_FILE = "faiss_index.bin"
CHUNKS_FILE = "chunks.pkl"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# =====================================================
# LOAD + CHUNK KNOWLEDGE BASE
# =====================================================

all_chunks = []

for file_path in glob.glob(os.path.join(DATA_DIR, "*.json")):

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    metadata = data.get("metadata", {})

    disease_id = metadata.get("disease_id", "UNKNOWN")
    disease_name = metadata.get("disease_name", disease_id)

    # =================================================
    # OVERVIEW
    # =================================================

    if "overview" in data:

        overview = data["overview"]

        text = (
            overview.get("definition", "")
            + " "
            + overview.get("summary", "")
        )

        all_chunks.append({
            "disease": disease_id,
            "type": "overview",
            "text": f"{disease_name}. {text}"
        })

    # =================================================
    # ETIOLOGY + RISK FACTORS
    # =================================================

    if "etiology" in data:

        etiology = data["etiology"]

        causes = etiology.get("causes", [])
        risks = etiology.get("risk_factors", [])

        all_chunks.append({
            "disease": disease_id,
            "type": "etiology",
            "text":
                f"{disease_name}. Causes: "
                + ", ".join(causes)
                + ". Risk factors: "
                + ", ".join(risks)
        })

    # =================================================
    # ABCDE RULE
    # =================================================

    if "clinical_features" in data:

        clinical = data["clinical_features"]

        abcde = clinical.get("abcde_rule", {})

        if len(abcde) > 0:

            all_chunks.append({
                "disease": disease_id,
                "type": "abcde_rule",
                "text":
                    f"{disease_name}. ABCDE rule: "
                    + ", ".join(abcde.values())
            })

    # =================================================
    # CLINICAL FEATURES
    # =================================================

    if "clinical_features" in data:

        clinical = data["clinical_features"]

        symptoms = clinical.get(
            "symptoms",
            []
        )

        visual = clinical.get(
            "visual_characteristics",
            {}
        )

        colors = visual.get("colors", [])
        shapes = visual.get("shapes", [])
        borders = visual.get("borders", [])

        all_chunks.append({
            "disease": disease_id,
            "type": "clinical_features",
            "text":
                f"{disease_name}. "
                + "Symptoms: "
                + ", ".join(symptoms)
                + ". Colors: "
                + ", ".join(colors)
                + ". Shapes: "
                + ", ".join(shapes)
                + ". Borders: "
                + ", ".join(borders)
        })

    # =================================================
    # DIAGNOSIS
    # =================================================

    if "diagnosis" in data:

        diagnosis = data["diagnosis"]

        diagnosis_items = []

        for value in diagnosis.values():

            if isinstance(value, list):
                diagnosis_items.extend(value)

        all_chunks.append({
            "disease": disease_id,
            "type": "diagnosis",
            "text":
                f"{disease_name}. Diagnosis: "
                + ", ".join(diagnosis_items)
        })

    # =================================================
    # TREATMENT
    # =================================================

    if "treatment" in data:

        treatment = data["treatment"]

        treatment_items = []

        for value in treatment.values():

            if isinstance(value, list):
                treatment_items.extend(value)

        all_chunks.append({
            "disease": disease_id,
            "type": "treatment",
            "text":
                f"{disease_name}. Treatment: "
                + ", ".join(treatment_items)
        })

    # =================================================
    # PROGNOSIS
    # =================================================

    if "prognosis" in data:

        prognosis = data["prognosis"]

        outlook = prognosis.get(
            "overall_outlook",
            ""
        )

        all_chunks.append({
            "disease": disease_id,
            "type": "prognosis",
            "text":
                f"{disease_name}. Prognosis: {outlook}"
        })

# =====================================================
# CHECK CHUNKS
# =====================================================

print("=" * 60)
print("TOTAL CHUNKS:", len(all_chunks))
print("=" * 60)

for chunk in all_chunks[:5]:
    print(chunk)
    print()

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

print("\nLoading embedding model...")

model = SentenceTransformer(
    EMBEDDING_MODEL
)

# =====================================================
# CREATE EMBEDDINGS
# =====================================================

texts = [
    chunk["text"]
    for chunk in all_chunks
]

embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

faiss.normalize_L2(embeddings)

embeddings = embeddings.astype(
    np.float32
)

print("\nEmbedding shape:", embeddings.shape)

# =====================================================
# BUILD FAISS INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    embeddings.shape[1]
)

index.add(
    embeddings
)

print(
    "\nVectors in FAISS:",
    index.ntotal
)

# =====================================================
# SAVE
# =====================================================

faiss.write_index(
    index,
    INDEX_FILE
)

with open(
    CHUNKS_FILE,
    "wb"
) as f:

    pickle.dump(
        all_chunks,
        f
    )

print("\nFAISS index saved.")

# =====================================================
# SEARCH FUNCTION
# =====================================================

def search(
    query,
    top_k=5
):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype(np.float32)

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    print("\n")
    print("=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for rank, idx in enumerate(
        indices[0],
        start=1
    ):

        chunk = all_chunks[idx]

        print(f"\nTOP {rank}")
        print("Disease :", chunk["disease"])
        print("Type    :", chunk["type"])
        print("Distance:", round(float(distances[0][rank - 1]), 4))
        print("Text    :", chunk["text"])

# =====================================================
# TEST
# =====================================================

search(
    "dark skin lesion with irregular border and color variation",
    top_k=5
)

TOTAL CHUNKS: 39
{'disease': 'AKIEC', 'type': 'overview', 'text': 'Actinic Keratosis. A precancerous lesion caused by chronic ultraviolet exposure. Actinic keratosis is a rough, scaly lesion occurring on sun-damaged skin and may progress to squamous cell carcinoma.'}

{'disease': 'AKIEC', 'type': 'etiology', 'text': 'Actinic Keratosis. Causes: Chronic UV exposure, Sun damage. Risk factors: Fair skin, Advanced age, Outdoor occupation, History of sunburn, Immunosuppression'}

{'disease': 'AKIEC', 'type': 'clinical_features', 'text': 'Actinic Keratosis. Symptoms: Itching, Burning, Bleeding, Crusting. Colors: Pink, Red, Brown. Shapes: . Borders: '}

{'disease': 'AKIEC', 'type': 'diagnosis', 'text': 'Actinic Keratosis. Diagnosis: Visual inspection, Scaly surface, Erythematous background, Atypical keratinocytes'}

{'disease': 'AKIEC', 'type': 'treatment', 'text': 'Actinic Keratosis. Treatment: Cryotherapy, Photodynamic therapy, 5-Fluorouracil, Imiquimod'}


Loading embedding model...

Embedd